In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv("covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop("has_covid", axis=1), df["has_covid"], test_size=0.2
)
X_train

,age,gender,fever,cough,city
92,82,Female,102.0,Strong,Kolkata
10,75,Female,NaN,Mild,Delhi
29,34,Female,NaN,Strong,Mumbai
63,10,Male,100.0,Mild,Bangalore
85,16,Female,103.0,Mild,Bangalore
...,...,...,...,...,...
90,59,Female,99.0,Strong,Delhi
21,73,Male,98.0,Mild,Bangalore
65,69,Female,102.0,Mild,Bangalore
16,69,Female,103.0,Mild,Kolkata


### 1. The Tedious Way

In [23]:
# Simple Imputer -> Fever
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[["fever"]])
X_test_fever = si.fit_transform(X_test[["fever"]])

In [24]:
# Ordinal Encoding -> Cough
oe = OrdinalEncoder(categories=[["Mild", "Strong"]])
X_train_cough = oe.fit_transform(X_train[["cough"]])

X_test_cough = oe.fit_transform(X_test[["cough"]])

In [25]:
# One Hot Encoding -> Gender, City
ohe = OneHotEncoder(drop="first", sparse_output=False)
X_train_gender_city = ohe.fit_transform(X_train[["gender", "city"]])

X_test_gender_city = ohe.fit_transform(X_test[["gender", "city"]])

In [26]:
# Extracing the remaining age column
X_train_age = X_train.drop(columns=["gender", "fever", "cough", "city"]).values

X_test_age = X_test.drop(columns=["gender", "fever", "cough", "city"]).values

In [27]:
# Final Concatenation

X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough), axis=1
)

X_test_transformed = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis=1
)

X_train_transformed.shape
X_test_transformed.shape

(20, 7)

### 2. Column Transformer

In [28]:
from sklearn.compose import ColumnTransformer

In [29]:
transformer = ColumnTransformer(
    transformers=(
        ("tnf1", SimpleImputer(), ["fever"]),
        ("tnf2", OrdinalEncoder(categories=[["Mild", "Strong"]]), ["cough"]),
        ("tnf3", OneHotEncoder(drop="first", sparse_output=False), ["gender", "city"]),
    ),
    remainder="passthrough",
)

In [30]:
transformer.fit_transform(X_train).shape
transformer.transform(X_test).shape

(20, 7)